# MEIKURAL — batch reports for handpicked performances

Run the cells in order with the **Python (MEIKURAL)** kernel. For **every audio file** in the test collection, this notebook displays and saves **one Matplotlib figure** containing:

- The filename, expected raag from its enclosing raag folder, predicted raag, and match/mismatch status.
- A line chart of all seven raag scores over time, exactly as in the single-performance notebook.
- The overall ranking using the existing **real-duration-weighted mean across that recording's successful windows**.

Folder labels are never passed into inference. Recordings are not pooled together. The original audio, training data, and checkpoint are not modified. Running the batch may take time; completed PNG reports remain available if you interrupt it.

**Version 2:** Sa is estimated from each entire recording once, not from individual chunks. Both confusion matrices use Version 2 predictions.

**Before use:** expand the Version 2 datasets, prepare features, and train `models/ver2/latest.pt` yourself. Version 1 checkpoints are rejected. Full-recording tonic estimation requires the whole supplied audio and can use substantial RAM; an incorrect estimate misaligns every window consistently.

In [ ]:
from pathlib import Path
import pickle
import sys

import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/ver2/config.json").is_file(), "Launch from the repository root or notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ver2.config import load_config, resolve_path
from src.ver2.inference.predict import load_model, predict
from src.ver2.inference.report import performance_figure, recording_labels

config = load_config()


## Paths

The defaults use the handpicked collection and trained Version 2 model. Set `DEVICE = "cuda"` to require GPU inference, `"cpu"` to force CPU, or leave `"auto"` to select the available device. Images are saved beneath `OUTPUT_DIR`, preserving the raag-folder structure.

In [ ]:
TEST_DIR = ROOT / "data/test_ver2_handpicked"
CHECKPOINT_PATH = resolve_path(config["paths"]["run"]) / "latest.pt"
OUTPUT_DIR = ROOT / "outputs/ver2/handpicked"
DEVICE = "auto"


## Inspect the recording list

Labels come from the first folder below `TEST_DIR`, including when files are nested more deeply. `sree` maps to `Shree`; underscores and hyphens act as spaces, so `miyan_malhar` maps to `Miyan Malhar`. Unknown labels remain unchanged and are marked as outside the model classes. In particular, **Bhairav is not renamed Bhairavi**.

In [ ]:
# Check the checkpoint and use its saved class names, independent of config edits.
model, checkpoint_config, _ = load_model(CHECKPOINT_PATH, device="cpu")
classes = checkpoint_config["classes"]
del model

TEST_DIR = Path(TEST_DIR).expanduser().resolve()
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
if OUTPUT_DIR == TEST_DIR or TEST_DIR in OUTPUT_DIR.parents or OUTPUT_DIR in TEST_DIR.parents:
    raise ValueError("Keep report images separate from the test collection")
recordings = recording_labels(TEST_DIR, classes)
print(f"Found {len(recordings)} recordings. Supported model classes: {', '.join(classes)}")
for audio_path, expected in recordings:
    print(f"{expected:15s} | {audio_path.relative_to(TEST_DIR)}")


## Run all recordings

This cell performs inference. Each recording gets a single combined figure, displayed inline and saved as a PNG. Scores on the right are taken directly from `predict()`; no new averaging formula is introduced. File names retain their audio extension in the PNG name to distinguish, for example, `performance.mp3.png` from `performance.wav.png`. Rerunning replaces reports at the same output paths.

A recording that fails receives an error figure, and the batch continues. Interrupting the kernel still stops the batch normally. Per-recording summaries remain in `summaries`; only per-window predicted labels are retained for the confusion matrices, rather than all per-window scores.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summaries = []
for index, (audio_path, expected) in enumerate(recordings, start=1):
    relative = audio_path.relative_to(TEST_DIR)
    print(f"[{index}/{len(recordings)}] {relative}", flush=True)
    error = None
    try:
        result = predict(audio_path, CHECKPOINT_PATH, device=DEVICE)
    except (OSError, ValueError, RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        result = None
        error = f"{type(exc).__name__}: {exc}"

    fig = performance_figure(result, relative.as_posix(), expected, error=error)
    image_path = OUTPUT_DIR / relative.parent / (relative.name + ".png")
    image_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(image_path, dpi=150, facecolor="white")
    display(fig)
    plt.close(fig)

    ranking = result["ranking"] if result is not None else []
    summaries.append({"audio": relative.as_posix(), "correct_raag": expected,
                      "predicted_raag": ranking[0]["raag"] if ranking else None,
                      "ranking": ranking, "image": str(image_path),
                      "chunk_predictions": [row["top_raag"] if row["scores"] is not None else None
                                            for row in result["windows"]] if result is not None else [],
                      "error": error or (result["message"] if not ranking else None)})
    print(f"Saved: {image_path}", flush=True)
print(f"Finished {len(summaries)} recording reports.")


## Interpreting the figures

The overall ranking is computed by the shared inference pipeline as

\[
\bar p_c = \frac{\sum_j d_j p_{j,c}}{\sum_j d_j},
\]

where \(d_j\) is the **real, unpadded audio duration** of successful window \(j\). This is the same mean as `05_ver2_inference.ipynb`, rather than an unweighted mean. Failed windows are excluded from the ranking and appear as gaps in the timeline. A performance with no usable windows shows no prediction.

Scores compare the seven supported raags; they are **not calibrated probabilities**. The expected label is supplied by the folder, not independently verified from the audio. A folder outside the supported classes is not silently relabelled. “Predicted raag” refers to one of the seven model classes, not a separate thaat/family classifier.

Overlap makes adjacent windows correlated. Incorrect tonic estimates, accompaniment, and unfamiliar artists can affect performance. Recordings used to guide model changes become development examples; keep independent recordings for a final test. See the PNGs in `outputs/ver2/handpicked/` for reports that can be shared without the notebook.

In [ ]:
# Song-level confusion matrix: each recording's final weighted-mean prediction is one vote.
from src.ver2.inference.report import confusion_figure

if "summaries" not in globals():
    raise RuntimeError("Run the batch inference cell first.")
song_samples = [row for row in summaries if row["predicted_raag"] is not None]
song_truth = [row["correct_raag"] for row in song_samples]
song_predictions = [row["predicted_raag"] for row in song_samples]
fig = confusion_figure(
    song_truth, song_predictions, classes,
    f"Song-level confusion matrix\n{len(song_samples)} songs counted | "
    f"{len(summaries) - len(song_samples)} songs without predictions excluded")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "confusion_matrix_songs.png", dpi=150, facecolor="white")
display(fig)
plt.close(fig)


In [ ]:
# Chunk-level confusion matrix: each successful model window is one unweighted vote.
# Includes zero-padded edge windows; overlapping windows are correlated.
from src.ver2.inference.report import confusion_figure

if "summaries" not in globals() or any("chunk_predictions" not in row for row in summaries):
    raise RuntimeError("Rerun the updated batch inference cell to retain every chunk's prediction.")
chunk_truth, chunk_predictions = [], []
failed_chunks = 0
for song in summaries:
    for prediction in song["chunk_predictions"]:
        if prediction is None:
            failed_chunks += 1
        else:
            chunk_truth.append(song["correct_raag"])
            chunk_predictions.append(prediction)
missing_chunk_recordings = sum(not row["chunk_predictions"] for row in summaries)
fig = confusion_figure(
    chunk_truth, chunk_predictions, classes,
    f"Chunk-level confusion matrix\n{len(chunk_predictions)} chunks counted | "
    f"{failed_chunks} failed chunks excluded\n"
    f"{missing_chunk_recordings} recordings had no chunk results")
fig.supxlabel("Each overlapping 30-second model window counts once, including padded edges.", fontsize=9)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / "confusion_matrix_chunks.png", dpi=150, facecolor="white")
display(fig)
plt.close(fig)
